<a href="https://colab.research.google.com/github/ShrutiKharate/Pytorch-MNIST-Four-MLP-Variant-/blob/main/pytorch_mnist_four_mlp_variants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST — Four MLP Variants (A/B/C/D)

**Goal:** Train and compare four fully‑connected neural networks (MLPs) on the MNIST dataset and analyze the impact of depth, width, normalization, activation, and regularization.

**Models**
- **Model A — Baseline (Shallow, Light):** `784 → 128 → 10`, ReLU, no BN, no dropout.
- **Model B — Deep + BN + Dropout:** `784 → 512 → 256 → 128 → 10`, ReLU, BatchNorm, Dropout=0.3.
- **Model C — Wide + GELU + strong Dropout:** `784 → 1024 → 10`, GELU, Dropout=0.5.
- **Model D — Compact + Reg + Label Smoothing:** `784 → 256 → 256 → 10`, ReLU, Dropout=0.4, AdamW (wd), Label smoothing=0.05.


## 1. Imports and Environment Checks

In [ ]:

# -----------------------------
# Standard library imports
# -----------------------------
import os  # provides OS utilities (paths, env, folders)
import math  # math helpers (may be useful)
import json  # saving results as JSON
import random  # Python RNG (for reproducibility)

# -----------------------------
# Third-party imports
# -----------------------------
import torch  # core PyTorch
import torch.nn as nn  # neural net modules
import torch.nn.functional as F  # functional ops: relu, ce, softmax, etc.
from torch.utils.data import DataLoader, random_split  # data pipeline helpers

from torchvision import datasets, transforms  # datasets (MNIST) + transforms
import torchvision  # top-level torchvision (optional)

import numpy as np  # numerical helpers
import matplotlib.pyplot as plt  # plotting (matplotlib only)

# -----------------------------
# Device selection
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # prefer GPU if available
print("Using device:", DEVICE)  # show chosen device


## 2. Reproducibility Setup

In [ ]:

def set_seed(seed: int = 42):
    """
    Set seeds for Python, NumPy, and PyTorch to encourage reproducible runs.
    """
    random.seed(seed)  # seed Python RNG
    np.random.seed(seed)  # seed NumPy RNG
    torch.manual_seed(seed)  # seed CPU tensors
    torch.cuda.manual_seed_all(seed)  # seed all GPU devices
    torch.backends.cudnn.deterministic = False  # allow non-deterministic but faster ops
    torch.backends.cudnn.benchmark = True  # let cuDNN find the fastest algorithms

set_seed(42)  # fix seed


## 3. Configuration (Paths, Hyperparameters)

In [ ]:

DATA_DIR = "./data"  # where to cache/download MNIST
RESULTS_DIR = "./results"  # where to save plots, weights, summaries
os.makedirs(RESULTS_DIR, exist_ok=True)  # create results directory if missing

BATCH_SIZE = 128  # batch size for training/eval
EPOCHS = 18  # epochs to train (increase to 20+ for slightly better results)
VAL_SPLIT = 0.10  # fraction of training set used as validation
LR_BASE = 1e-3  # base LR for AdamW-based setups

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("BATCH_SIZE:", BATCH_SIZE, "EPOCHS:", EPOCHS, "VAL_SPLIT:", VAL_SPLIT)


## 4. Dataset and DataLoaders

In [ ]:

# Define transforms: convert to tensor and normalize with MNIST stats
transform = transforms.Compose([
    transforms.ToTensor(),  # HxW in [0,255] -> CxHxW in [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # standard MNIST normalization
])

# Download/load MNIST
train_full = datasets.MNIST(root=DATA_DIR, train=True, transform=transform, download=True)  # 60k train
test_set = datasets.MNIST(root=DATA_DIR, train=False, transform=transform, download=True)  # 10k test

# Split a validation set from training set
val_len = int(len(train_full) * VAL_SPLIT)  # number of validation samples
train_len = len(train_full) - val_len  # remaining for training
train_set, val_set = random_split(train_full, [train_len, val_len], generator=torch.Generator().manual_seed(42))

# Create DataLoaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

len(train_set), len(val_set), len(test_set)


## 5. Utility Functions

In [ ]:

def count_parameters(model: nn.Module) -> int:
    """Return the number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)  # sum sizes of trainable tensors

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Compute accuracy from logits and ground-truth labels."""
    preds = logits.argmax(dim=1)  # predicted class index per sample
    return (preds == y).float().mean().item()  # average correctness as float

@torch.no_grad()
def make_confusion_matrix(model: nn.Module, loader: DataLoader, device=DEVICE, num_classes=10):
    """Compute confusion matrix (rows=true, cols=pred)."""
    cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)  # initialize counts
    model.eval()  # eval mode (no dropout, BN uses running stats)
    for x, y in loader:  # over batches
        x, y = x.to(device), y.to(device)  # move to device
        logits = model(x)  # forward pass
        preds = logits.argmax(dim=1)  # predicted labels
        for t, p in zip(y.view(-1), preds.view(-1)):  # iterate sample-wise
            cm[t.long(), p.long()] += 1  # increment cell
    return cm.cpu().numpy()  # return as NumPy array

def per_class_accuracy(cm):
    """Return dict: class -> per-class accuracy derived from confusion matrix."""
    acc = {}
    for i in range(cm.shape[0]):
        correct = cm[i, i]  # true positives for class i
        total = cm[i].sum()  # total true samples for class i
        acc[i] = float(correct) / float(total) if total > 0 else 0.0  # guard against zero
    return acc


## 6. MLP Model (Configurable Depth/Width/Activation/BN/Dropout)

In [ ]:

class MLP(nn.Module):
    def __init__(self, hidden_sizes, activation="relu", use_batchnorm=False, dropout=0.0):
        super().__init__()  # init nn.Module

        # activation factory
        act_layer = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU,
            "leakyrelu": lambda: nn.LeakyReLU(0.1)
        }[activation]

        layers = []  # accumulating layers
        in_features = 28 * 28  # MNIST flattened size

        for h in hidden_sizes:  # build hidden stack
            layers.append(nn.Linear(in_features, h))  # linear layer
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))  # optional BN
            layers.append(act_layer())  # activation
            if dropout > 0:
                layers.append(nn.Dropout(dropout))  # optional dropout
            in_features = h  # next in_features

        layers.append(nn.Linear(in_features, 10))  # output layer (10 classes)
        self.net = nn.Sequential(*layers)  # wrap as Sequential

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten (N,1,28,28) -> (N,784)
        return self.net(x)  # pass through layers


## 7. Cross-Entropy with Label Smoothing (used by Model D)

In [ ]:

class CrossEntropyLabelSmoothing(nn.Module):
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        assert 0.0 <= smoothing < 1.0, "Smoothing must be in [0,1)."
        self.smoothing = smoothing  # store epsilon

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        if self.smoothing == 0.0:  # fallback to standard CE
            return F.cross_entropy(logits, target)
        n_classes = logits.size(-1)  # number of classes
        log_probs = F.log_softmax(logits, dim=-1)  # log-softmax for stability
        with torch.no_grad():  # build smoothed targets without grads
            true_dist = torch.zeros_like(log_probs)  # init zeros
            true_dist.fill_(self.smoothing / (n_classes - 1))  # distribute epsilon to non-true classes
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)  # put 1-eps at true label
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))  # mean CE over batch


## 8. Training and Evaluation Loops

In [ ]:

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()  # train mode
    epoch_loss = 0.0  # accumulate loss
    epoch_acc = 0.0  # accumulate correct
    for x, y in loader:  # each batch
        x, y = x.to(device), y.to(device)  # move to device
        optimizer.zero_grad(set_to_none=True)  # clear grads
        logits = model(x)  # forward
        loss = criterion(logits, y)  # loss
        loss.backward()  # backward
        optimizer.step()  # step
        epoch_loss += loss.item() * x.size(0)  # sum loss over samples
        epoch_acc += (logits.argmax(1) == y).float().sum().item()  # sum correct
    n = len(loader.dataset)  # number of samples
    return epoch_loss / n, epoch_acc / n  # averages

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # eval mode
    epoch_loss = 0.0
    epoch_acc = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        epoch_loss += loss.item() * x.size(0)
        epoch_acc += (logits.argmax(1) == y).float().sum().item()
    n = len(loader.dataset)
    return epoch_loss / n, epoch_acc / n


## 9. Learning-Rate Schedulers

In [ ]:

def build_scheduler(sched_name, optimizer, epochs):
    if sched_name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(epochs // 3, 1), gamma=0.3)
    elif sched_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    else:
        return None


## 10. Experiment Configurations (A/B/C/D)

In [ ]:

from dataclasses import dataclass
from typing import Tuple

@dataclass
class ExperimentConfig:
    name: str
    hidden: Tuple[int, ...]
    activation: str
    batchnorm: bool
    dropout: float
    optimizer: str
    lr: float
    weight_decay: float
    scheduler: str
    label_smoothing: float

EXPERIMENTS = [
    ExperimentConfig("A_Baseline_Shallow_ReLU",
                     hidden=(128,), activation="relu", batchnorm=False, dropout=0.0,
                     optimizer="sgd", lr=0.05, weight_decay=0.0, scheduler="step",
                     label_smoothing=0.0),
    ExperimentConfig("B_Deep_BN_Dropout_ReLU",
                     hidden=(512, 256, 128), activation="relu", batchnorm=True, dropout=0.3,
                     optimizer="adamw", lr=1e-3, weight_decay=1e-4, scheduler="cosine",
                     label_smoothing=0.0),
    ExperimentConfig("C_Wide_GELU_Dropout",
                     hidden=(1024,), activation="gelu", batchnorm=False, dropout=0.5,
                     optimizer="adamw", lr=1e-3, weight_decay=1e-5, scheduler="step",
                     label_smoothing=0.0),
    ExperimentConfig("D_Compact_Reg_LabelSmooth",
                     hidden=(256, 256), activation="relu", batchnorm=False, dropout=0.4,
                     optimizer="adamw", lr=1e-3, weight_decay=5e-4, scheduler="cosine",
                     label_smoothing=0.05),
]


## 11. Optimizer Builder and Plotting

In [ ]:

def build_optimizer(opt_name, params, lr, weight_decay):
    if opt_name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=weight_decay)
    elif opt_name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {opt_name}")

def plot_curves(history, title, out_prefix):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    plt.figure()
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} — Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{out_prefix}_loss.png", dpi=140)
    plt.close()

    plt.figure()
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} — Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{out_prefix}_acc.png", dpi=140)
    plt.close()


## 12. Run a Single Experiment (Train→Validate→Test→Save Artifacts)

In [ ]:

def run_experiment(cfg):
    print(f"\n=== Running {cfg.name} ===")
    model = MLP(cfg.hidden, activation=cfg.activation, use_batchnorm=cfg.batchnorm, dropout=cfg.dropout).to(DEVICE)
    n_params = count_parameters(model)
    print(f"Params: {n_params:,}")
    criterion = CrossEntropyLabelSmoothing(smoothing=cfg.label_smoothing)
    optimizer = build_optimizer(cfg.optimizer, model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = build_scheduler(cfg.scheduler, optimizer, EPOCHS)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val = -1.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)
        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        print(f"Epoch {epoch:02d}: train_loss={tr_loss:.4f}  train_acc={tr_acc*100:5.2f}% | "
              f"val_loss={va_loss:.4f}    val_acc={va_acc*100:5.2f}%")

        if va_acc > best_val:
            best_val = va_acc
            best_state = {"model": model.state_dict(), "epoch": epoch, "val_acc": va_acc}

    if best_state is not None:
        model.load_state_dict(best_state["model"])

    te_loss, te_acc = evaluate(model, test_loader, criterion, DEVICE)
    print(f"[{cfg.name}] Test: loss={te_loss:.4f}  acc={te_acc*100:.2f}%")

    cm = make_confusion_matrix(model, test_loader, device=DEVICE, num_classes=10)
    pc = per_class_accuracy(cm)

    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{cfg.name}.pt"))
    with open(os.path.join(RESULTS_DIR, f"{cfg.name}_per_class.json"), "w") as f:
        json.dump(pc, f, indent=2)

    plot_curves(history, cfg.name, os.path.join(RESULTS_DIR, f"{cfg.name}"))

    plt.figure(figsize=(5.2, 4.6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{cfg.name} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{cfg.name}_cm.png"), dpi=140)
    plt.close()

    return {"name": cfg.name, "params": int(n_params), "best_val_acc": float(best_val),
            "test_acc": float(te_acc), "test_loss": float(te_loss), "per_class_acc": pc}


## 13. Run All Experiments and Summarize

In [ ]:

summaries = []
for cfg in EXPERIMENTS:
    summaries.append(run_experiment(cfg))

print("\n=== Summary ===")
header = f"{'Model':34} | {'Params':>10} | {'Best Val Acc':>12} | {'Test Acc':>8}"
print(header)
print("-" * len(header))
for r in summaries:
    print(f"{r['name'][:34]:34} | {r['params']:10,d} | {r['best_val_acc']*100:12.2f}% | {r['test_acc']*100:8.2f}%")

summary_path = os.path.join(RESULTS_DIR, "summary.json")
with open(summary_path, "w") as f:
    json.dump(summaries, f, indent=2)
print("Summary written to:", summary_path)


## 14. Tips and Expected Ranges
- Model A: ~97.5–98.5%
- Model B: ~98.8–99.3%
- Model C: ~98.8–99.1%
- Model D: ~98.5–99.0%

If you are offline, the `download=True` in the MNIST loader may fail; in that case pre‑download MNIST to `DATA_DIR`.